In [1]:
library(Matrix)
library(cytosignal)
library(parallel)

In [2]:
setwd('/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC')

In [5]:

info_df = read.csv("Table/Patient_Info.csv",stringsAsFactors = FALSE)
sample_map <- unique(info_df[, c("SampleName", "SampleID")])

cyto_obj <- readRDS( "Data/ST/CytoSignal/cyto_recep.rds")


In [6]:

process_one_sample <- function(i) {
    sample_name <- sample_map$SampleName[i]
    sample_id   <- sample_map$SampleID[i]  
    
    message(">>> Processing: ", sample_name, " | ", sample_id)
    
    in_dir <- file.path("Data/ST/Samples", sample_name, "cytoSignal")
 
    required_files <- c("counts.mtx", "genes.tsv", "barcodes.tsv", "spatials.csv", "cluster.tsv")
    missing_files <- required_files[!file.exists(file.path(in_dir, required_files))]
    
    if (length(missing_files) > 0) {
        stop("Missing files: ", paste(missing_files, collapse = ", "))
    }
    
    tryCatch({
        
        m <- readMM(file.path(in_dir, "counts.mtx"))
        genes <- readLines(file.path(in_dir, "genes.tsv"))
        cells <- readLines(file.path(in_dir, "barcodes.tsv"))
        
        rownames(m) <- genes
        colnames(m) <- cells
        
        spatial <- read.csv(file.path(in_dir, "spatials.csv"), header = TRUE, row.names = 1, check.names = FALSE)
        spatial <- spatial[cells, c("x", "y"), drop = FALSE]
        spatial <- as.matrix(spatial)
        
        cluster_df <- read.table(file.path(in_dir, "cluster.tsv"), sep = "\t", header = TRUE, row.names = 1, check.names = FALSE)
        cluster_df <- subset(cluster_df,!is.na(CellType)&CellType != 'conflict' & GLMPCA_leiden != 'conflict')
        
        cluster <- cluster_df$CellType
        cluster <- factor(cluster)
        cells <- rownames(cluster_df)
        names(cluster) <- cells
        
        cs <- createCytoSignal(raw.data = m[,cells], cells.loc = spatial[cells,], clusters = cluster)
        cs <- addIntrDB(cs, cyto_obj$g2u, cyto_obj$db.diff, cyto_obj$db.cont, cyto_obj$inter.index)
        cs <- removeLowQuality(cs, counts.thresh = 100)
        cs <- changeUniprot(cs)
        cs <- inferEpsParams(cs, scale.factor = 1)
        cs <- findNN(cs)
        cs <- imputeLR(cs)
        saveRDS(cs, file = file.path(in_dir, "cytosignal.rds"))
        
        n_cells <- ncol(cs@raw.counts)
        perm_use <- min(500, n_cells)
        set.seed(123 + sum(utf8ToInt(sample_id)))
        cs <- inferIntrScore(cs, recep.smooth = TRUE, perm.size = perm_use, numCores = 1)
        cs <- inferSignif(cs, p.thresh = 0.01, reads.thresh = 10, sig.thresh = 10)
        cs <- rankIntrSpatialVar(cs, numCores = 1)
        saveRDS(cs, file =  file.path(in_dir, "cytosignal_add_LR_score.rds"))
  
        done_msg <- paste0("[DONE] ", sample_name, " | ", sample_id, " | n_cells = ", n_cells)
        return(data.frame(SampleName = sample_name,
                          SampleID = sample_id,
                          status = "done",
                          n_cells = n_cells,
                          msg = done_msg,
                          stringsAsFactors = FALSE))
        
    }, error = function(e) {
        message("[ERROR] ", sample_name, " | ", sample_id, " : ", e$message)
        return(data.frame(SampleName = sample_name,
                          SampleID = sample_id,
                          status = "error",
                          n_cells = NA,
                          msg = e$message,
                          stringsAsFactors = FALSE))
    })
}



In [ ]:
res_list <- mclapply(seq_len(nrow(sample_map))[52:54], process_one_sample, mc.cores = 16)
res_df <- do.call(rbind, res_list)